In [17]:
from datetime import date
from pathlib import Path
import re

import pandas as pd

# ---- User settings ---------------------------------------------------------
PROJECT_ROOT = Path('..').resolve() if Path.cwd().name == 'code' else Path('.').resolve()
DATA_ROOT = PROJECT_ROOT / 'data'

# Set to None or 'all' to scan every data/merged* folder.
# Set to a specific folder to scan only that dataset, for example:
#   SEQKIT_INPUT_FOLDER = 'mergedseq5'
#   SEQKIT_INPUT_FOLDER = 'data/mergedseq5'
#   SEQKIT_INPUT_FOLDER = '/absolute/path/to/data/mergedseq5'
SEQKIT_INPUT_FOLDER = 'data/mergedseq5'

# Raw seqkit locate files created by code/run_primer_umi_tracking.sh.
SEQKIT_HITS_GLOB = 'barcode*_primer_hits.tsv'

OUTPUT_DIR = DATA_ROOT / 'primertrack'


def resolve_search_root(folder):
    if folder is None or str(folder).strip().lower() in {'', 'all'}:
        return DATA_ROOT
    folder = Path(folder)
    if folder.is_absolute():
        return folder
    if folder.parts and folder.parts[0] == 'data':
        return PROJECT_ROOT / folder
    return DATA_ROOT / folder


SEQKIT_SEARCH_ROOT = resolve_search_root(SEQKIT_INPUT_FOLDER)


def output_dataset_label(search_root):
    search_root = Path(search_root)
    if search_root == DATA_ROOT:
        return 'all'
    if search_root.name.startswith('barcode'):
        return search_root.parent.name
    return search_root.name


RUN_DATE = date.today().strftime('%Y%m%d')
SEQKIT_OUTPUT_LABEL = output_dataset_label(SEQKIT_SEARCH_ROOT)
OUTPUT_BEST_HITS = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hits_{RUN_DATE}.txt'
OUTPUT_ALL_HITS = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_all_hits_with_metadata_{RUN_DATE}.txt'
OUTPUT_COUNTS_BY_BARCODE = OUTPUT_DIR / f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hit_counts_by_barcode_{RUN_DATE}.txt'

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Seqkit search root: {SEQKIT_SEARCH_ROOT}')
print(f'Output label: {SEQKIT_OUTPUT_LABEL}')
print(f'Run date: {RUN_DATE}')
print(f'Output best-hit table: {OUTPUT_BEST_HITS}')

Project root: /mnt/c/Users/mftor/Documents/metabarcoding
Data root: /mnt/c/Users/mftor/Documents/metabarcoding/data
Seqkit search root: /mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5
Output label: mergedseq5
Run date: 20260630
Output best-hit table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt


In [18]:
# Discover raw seqkit locate outputs.
if not SEQKIT_SEARCH_ROOT.exists():
    raise FileNotFoundError(f'Seqkit search root does not exist: {SEQKIT_SEARCH_ROOT}')

if SEQKIT_SEARCH_ROOT == DATA_ROOT:
    seqkit_files = sorted(DATA_ROOT.glob(f'merged*/barcode*/{SEQKIT_HITS_GLOB}'))
elif SEQKIT_SEARCH_ROOT.name.startswith('barcode'):
    seqkit_files = sorted(SEQKIT_SEARCH_ROOT.glob(SEQKIT_HITS_GLOB))
else:
    seqkit_files = sorted(SEQKIT_SEARCH_ROOT.glob(f'barcode*/{SEQKIT_HITS_GLOB}'))

print(f'Seqkit output files found: {len(seqkit_files)}')
for path in seqkit_files[:10]:
    print(path)

if not seqkit_files:
    raise FileNotFoundError(f'No files matching {SEQKIT_HITS_GLOB!r} found under {SEQKIT_SEARCH_ROOT}')

Seqkit output files found: 24
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode01/barcode01_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode02/barcode02_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode03/barcode03_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode04/barcode04_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode05/barcode05_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode06/barcode06_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode07/barcode07_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode08/barcode08_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode09/barcode09_primer_hits.tsv
/mnt/c/Users/mftor/Documents/metabarcoding/data/mergedseq5/barcode10/barcode10_primer_hits.tsv


In [19]:
IUPAC = {
    'A': set('A'),
    'C': set('C'),
    'G': set('G'),
    'T': set('T'),
    'U': set('T'),
    'R': set('AG'),
    'Y': set('CT'),
    'S': set('GC'),
    'W': set('AT'),
    'K': set('GT'),
    'M': set('AC'),
    'B': set('CGT'),
    'D': set('AGT'),
    'H': set('ACT'),
    'V': set('ACG'),
    'N': set('ACGT'),
}


def path_metadata(path):
    path = Path(path)
    parts = path.parts
    # returns the index for the path part that contains 'barcode' else returns None
    barcode_index = next((i for i, part in enumerate(parts) if part.startswith('barcode')), None)
    barcode_folder = parts[barcode_index] if barcode_index is not None else None
    merged_folder = parts[barcode_index - 1] if barcode_index is not None and barcode_index > 0 else None
    return merged_folder, barcode_folder


def normalize_seqkit_columns(df):
    # seqkit locate commonly writes: seqID, patternName, pattern, strand, start, end, matched.
    # Keep any additional columns if present.
    df.columns = [str(col).strip() for col in df.columns]
    rename = {}
    for col in df.columns:
        low = col.lower()
        if low in {'seqid', 'seq_id', 'read_id', 'name'}:
            rename[col] = 'seqID'
        elif low in {'patternname', 'pattern_name'}:
            rename[col] = 'patternName'
    return df.rename(columns=rename)


def iupac_mismatches(pattern, matched):
    if pd.isna(pattern) or pd.isna(matched):
        return pd.NA
    pattern = str(pattern).upper().replace('U', 'T')
    matched = str(matched).upper().replace('U', 'T')
    # length of the shortest sequence between pattern and matched, i.e. the matched
    length = min(len(pattern), len(matched))
    mismatches = abs(len(pattern) - len(matched))
    # sequence parts that could mathc or mismatch, to count matches ignoring ambiguous nucleotides
    matches = 0
    # estimate mismatches accounting for different lengths between pattern and match
    for p, m in zip(pattern[:length], matched[:length]):
        allowed = IUPAC.get(p, IUPAC['N'])
        # does not count N or ambiguous as a match
        if m in allowed:
            matches += 1
        else:
            mismatches += 1
    return mismatches

# calculates the non-ambiguous comparable nucleotides in pattern (primers are degenerated)
def informative_length(pattern):
    if pd.isna(pattern):
        return pd.NA
    pattern = str(pattern).upper().replace('U', 'T')
    informative = sum(1 for base in pattern if base != 'N')
    return informative or len(pattern)

# annotates file data to keep track of barcode, dataset, and experiment
def read_seqkit_file(path):
    df = pd.read_csv(path, sep='\t')
    df = normalize_seqkit_columns(df)
    merged_folder, barcode_folder = path_metadata(path)
    df['merged_folder'] = merged_folder
    df['barcode_folder'] = barcode_folder
    df['source_seqkit_tsv'] = str(path)
    return df

In [20]:
# Read all raw seqkit locate outputs and add metadata columns.
seqkit_tables = [read_seqkit_file(path) for path in seqkit_files]
all_hits = pd.concat(seqkit_tables, ignore_index=True) if seqkit_tables else pd.DataFrame()

print(f'Raw seqkit hit rows: {len(all_hits):,}')
print(f'Columns: {list(all_hits.columns)}')
all_hits.head()

Raw seqkit hit rows: 13,106,750
Columns: ['seqID', 'patternName', 'pattern', 'strand', 'start', 'end', 'matched', 'merged_folder', 'barcode_folder', 'source_seqkit_tsv']


,seqID,patternName,pattern,strand,start,end,matched,merged_folder,barcode_folder,source_seqkit_tsv
0,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_90,ggtagtctgtatctctatgtgrktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
1,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_80,ggtagtatatatgtctatagarktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
2,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_76,ggtagtagagagcgtcgcgtgrktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
3,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,ggtagtatagagctctacatarktcaacmaatcataaagatattgg,+,81,126,ggtagtatagagctctacataattcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...
4,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,ggtagtatagagctctacatarktcaacmaatcataaagatattgg,-,251,296,ggtagtatagagctctacataattcaaccaatcataacagtatcgt,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...


In [21]:
# Compute score columns used only for sorting best hits.
# Lower mismatch_count is better. Higher primer_identity and matched_length are better.
required = {'seqID', 'pattern', 'matched'}
missing = required - set(all_hits.columns)
if missing:
    raise ValueError(f'Seqkit table is missing required columns for best-hit scoring: {sorted(missing)}')

all_hits['mismatch_count'] = [iupac_mismatches(p, m) for p, m in zip(all_hits['pattern'], all_hits['matched'])]
all_hits['informative_pattern_length'] = [informative_length(p) for p in all_hits['pattern']]
all_hits['matched_length'] = all_hits['matched'].astype(str).str.len()
all_hits['primer_identity'] = 1 - (all_hits['mismatch_count'] / all_hits['informative_pattern_length'])

# Preserve a complete raw table as a separate output with metadata.
all_hits.to_csv(OUTPUT_ALL_HITS, sep='\t', index=False)
print(f'Wrote all-hit table with metadata: {OUTPUT_ALL_HITS}')

all_hits[['merged_folder', 'barcode_folder', 'seqID', 'patternName', 'mismatch_count', 'primer_identity', 'matched_length']].head()

Wrote all-hit table with metadata: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_all_hits_with_metadata_20260630.txt


,merged_folder,barcode_folder,seqID,patternName,mismatch_count,primer_identity,matched_length
0,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_90,7,0.847826,46
1,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_80,5,0.891304,46
2,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_76,7,0.847826,46
3,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,0,1.000000,46
4,mergedseq5,barcode01,3824751f-f959-4e10-a863-cee50e654250,forward_tagged_primer_78,5,0.891304,46


In [22]:
# Keep the best forward hit and best reverse hit per read within each merged*/barcode* folder.
# Aiming for:
#   1. lowest mismatch_count
#   2. highest primer_identity
#   3. highest matched_length
#   4. earliest start, if the start column exists, in case random matches happen mid-read (unexpected unless ligations went wrong)
#   5. stable patternName order, if two best hits have all equal but pattern name, it sorts alphabetically (BEWARE THIS IS NOT EXPERIMENTALLY COHERENT) but if all else equal, there's no much to do but to check manually the frequency of identical reads with one or the other primer+umi
# needed as otherwise the selected "best" hit will be whichever appeared first with low reproducibility (adds determinism)
all_hits['primer_direction'] = (
    all_hits['patternName']
    .astype('string')
    .str.extract(r'^(forward|reverse)(?:_|$)', expand=False)
)
unknown_direction = all_hits['primer_direction'].isna()
if unknown_direction.any():
    unknown_patterns = sorted(all_hits.loc[unknown_direction, 'patternName'].dropna().astype(str).unique())
    print(f'Ignoring {unknown_direction.sum():,} hits whose patternName does not start with forward or reverse: {unknown_patterns}')

directional_hits = all_hits.loc[~unknown_direction].copy()
sort_columns = ['merged_folder', 'barcode_folder', 'seqID', 'primer_direction', 'mismatch_count', 'primer_identity', 'matched_length']
ascending = [True, True, True, True, True, False, False]

# appending to the list of columns to sort, at the end of the list (lowest priority for selecting best hit)
if 'start' in directional_hits.columns:
    sort_columns.append('start')
    ascending.append(True)
if 'patternName' in directional_hits.columns:
    sort_columns.append('patternName')
    ascending.append(True)

# mergesort is stable. It preserves the original order of rows when the sort keys are tied
sorted_hits = directional_hits.sort_values(sort_columns, ascending=ascending, kind='mergesort')
best_hits_long = sorted_hits.drop_duplicates(
    ['merged_folder', 'barcode_folder', 'seqID', 'primer_direction'],
    keep='first',
).copy()

read_keys = ['merged_folder', 'barcode_folder', 'seqID']
best_hits = None
for direction in ['forward', 'reverse']:
    direction_hits = best_hits_long.loc[best_hits_long['primer_direction'].eq(direction)].drop(columns='primer_direction')
    direction_hits = direction_hits.rename(
        columns={column: f'{direction}_{column}' for column in direction_hits.columns if column not in read_keys}
    )
    best_hits = direction_hits if best_hits is None else best_hits.merge(direction_hits, on=read_keys, how='outer', validate='one_to_one')

best_hits = best_hits.sort_values(read_keys, kind='mergesort').reset_index(drop=True)

best_hits.to_csv(OUTPUT_BEST_HITS, sep='\t', index=False)

print(f'Reads with at least one directional best hit: {len(best_hits):,}')
print(f'Forward best hits: {best_hits["forward_patternName"].notna().sum():,}')
print(f'Reverse best hits: {best_hits["reverse_patternName"].notna().sum():,}')
print(f'Wrote best-hit table: {OUTPUT_BEST_HITS}')
best_hits.head()

Reads with at least one directional best hit: 3,727,055
Forward best hits: 2,845,997
Reverse best hits: 2,522,123
Wrote best-hit table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt


,seqID,forward_patternName,forward_pattern,forward_strand,forward_start,forward_end,forward_matched,merged_folder,barcode_folder,forward_source_seqkit_tsv,...,reverse_pattern,reverse_strand,reverse_start,reverse_end,reverse_matched,reverse_source_seqkit_tsv,reverse_mismatch_count,reverse_informative_pattern_length,reverse_matched_length,reverse_primer_identity
0,0000bc31-cdb1-4513-8018-c453ae7b47ba,forward_tagged_primer_24,ggtagcacacgcgcgctatatrktcaacmaatcataaagatattgg,+,71.0,116.0,ggtagcacacgcgcgctatatgttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000f46e-1613-40e5-93ca-2c312d61b67a,NaN,NaN,NaN,NaN,NaN,NaN,mergedseq5,barcode01,NaN,...,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,+,113.0,159.0,ggtagatatatgcacgctctataaacttctggatgtccaaaaaatca,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,0.0,47.0,47.0,1.0
2,0000f966-3c52-46f4-9573-bf8c8a5efcdf,forward_tagged_primer_55,ggtaggagactagagatagtgrktcaacmaatcataaagatattgg,-,111.0,156.0,ggtaggagactagagatagtggttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000122a7-5b12-4988-99a7-bdc6c540c2a3,forward_tagged_primer_46,ggtagctcacactctctcacarktcaacmaatcataaagatattgg,+,82.0,127.0,ggtagctcacactctctcacagttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016a97-d608-4421-887d-297dc3985c89,forward_tagged_primer_5,ggtagacacatatcgcactacrktcaacmaatcataaagatattgg,+,64.0,109.0,ggtagacacatctcgcactacagtcaaccaatcatgaaggatcttg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [23]:
# Quick summary by merged folder, barcode, and primer+umi.
summary = (
    best_hits_long
    .groupby(['merged_folder', 'barcode_folder', 'patternName'], dropna=False)
    .size()
    .reset_index(name='read_count')
    .sort_values(['merged_folder', 'barcode_folder', 'read_count'], ascending=[True, True, False])
)

summary_path = OUTPUT_COUNTS_BY_BARCODE
summary.to_csv(summary_path, sep='\t', index=False)
print(f'Wrote summary table: {summary_path}')
summary.head(20)

Wrote summary table: /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hit_counts_by_barcode_20260630.txt


,merged_folder,barcode_folder,patternName,read_count
97,mergedseq5,barcode01,reverse_tagged_primer_2,278598
79,mergedseq5,barcode01,forward_tagged_primer_81,11532
12,mergedseq5,barcode01,forward_tagged_primer_20,10019
84,mergedseq5,barcode01,forward_tagged_primer_86,9872
75,mergedseq5,barcode01,forward_tagged_primer_78,8333
93,mergedseq5,barcode01,forward_tagged_primer_94,8066
87,mergedseq5,barcode01,forward_tagged_primer_89,7925
28,mergedseq5,barcode01,forward_tagged_primer_35,7875
82,mergedseq5,barcode01,forward_tagged_primer_84,7778
29,mergedseq5,barcode01,forward_tagged_primer_36,7528


In [24]:
# Reload this notebook's generated outputs for visualization without rerunning seqkit parsing.
# Run the first settings cell first, then you can continue with this cell.

OUTPUT_RELOAD_OVERRIDE = {
    'all_hits': None,
    'best_hits': None,
    'counts_by_barcode': None,
}


def find_single_output(path, glob_pattern, description):
    path = Path(path)
    if path.exists():
        return path

    matches = sorted(OUTPUT_DIR.glob(glob_pattern))
    if not matches:
        raise FileNotFoundError(
            f'Could not find {description}. Expected {path} or one file matching {glob_pattern}'
        )
    if len(matches) > 1:
        match_list = '\n'.join(str(match) for match in matches)
        raise ValueError(
            f'Multiple files matched {description}. Keep only the intended file in {OUTPUT_DIR} '
            f'or set OUTPUT_RELOAD_OVERRIDE explicitly:\n{match_list}'
        )
    return matches[0]


reload_all_hits_path = Path(OUTPUT_RELOAD_OVERRIDE['all_hits']) if OUTPUT_RELOAD_OVERRIDE['all_hits'] else find_single_output(
    OUTPUT_ALL_HITS,
    f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_all_hits_with_metadata_????????.txt',
    'all-hit table',
)
reload_best_hits_path = Path(OUTPUT_RELOAD_OVERRIDE['best_hits']) if OUTPUT_RELOAD_OVERRIDE['best_hits'] else find_single_output(
    OUTPUT_BEST_HITS,
    f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hits_????????.txt',
    'best-hit table',
)
reload_counts_path = Path(OUTPUT_RELOAD_OVERRIDE['counts_by_barcode']) if OUTPUT_RELOAD_OVERRIDE['counts_by_barcode'] else find_single_output(
    OUTPUT_COUNTS_BY_BARCODE,
    f'{SEQKIT_OUTPUT_LABEL}_seqkit_primer_umi_best_hit_counts_by_barcode_????????.txt',
    'best-hit counts by barcode table',
)

all_hits = pd.read_csv(reload_all_hits_path, sep='\t')
best_hits = pd.read_csv(reload_best_hits_path, sep='\t')
summary = pd.read_csv(reload_counts_path, sep='\t')

print(f'Reloaded all_hits: {len(all_hits):,} rows from {reload_all_hits_path}')
print(f'Reloaded best_hits: {len(best_hits):,} rows from {reload_best_hits_path}')
print(f'Reloaded summary: {len(summary):,} rows from {reload_counts_path}')

best_hits.head()


Reloaded all_hits: 13,106,750 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_all_hits_with_metadata_20260630.txt
Reloaded best_hits: 3,727,055 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hits_20260630.txt
Reloaded summary: 2,356 rows from /mnt/c/Users/mftor/Documents/metabarcoding/data/primertrack/mergedseq5_seqkit_primer_umi_best_hit_counts_by_barcode_20260630.txt


,seqID,forward_patternName,forward_pattern,forward_strand,forward_start,forward_end,forward_matched,merged_folder,barcode_folder,forward_source_seqkit_tsv,...,reverse_pattern,reverse_strand,reverse_start,reverse_end,reverse_matched,reverse_source_seqkit_tsv,reverse_mismatch_count,reverse_informative_pattern_length,reverse_matched_length,reverse_primer_identity
0,0000bc31-cdb1-4513-8018-c453ae7b47ba,forward_tagged_primer_24,ggtagcacacgcgcgctatatrktcaacmaatcataaagatattgg,+,71.0,116.0,ggtagcacacgcgcgctatatgttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0000f46e-1613-40e5-93ca-2c312d61b67a,NaN,NaN,NaN,NaN,NaN,NaN,mergedseq5,barcode01,NaN,...,ggtagatatatgcacgctctataaacttcwggrtgwccaaaaaatca,+,113.0,159.0,ggtagatatatgcacgctctataaacttctggatgtccaaaaaatca,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,0.0,47.0,47.0,1.0
2,0000f966-3c52-46f4-9573-bf8c8a5efcdf,forward_tagged_primer_55,ggtaggagactagagatagtgrktcaacmaatcataaagatattgg,-,111.0,156.0,ggtaggagactagagatagtggttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000122a7-5b12-4988-99a7-bdc6c540c2a3,forward_tagged_primer_46,ggtagctcacactctctcacarktcaacmaatcataaagatattgg,+,82.0,127.0,ggtagctcacactctctcacagttcaaccaatcataaagatattgg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,00016a97-d608-4421-887d-297dc3985c89,forward_tagged_primer_5,ggtagacacatatcgcactacrktcaacmaatcataaagatattgg,+,64.0,109.0,ggtagacacatctcgcactacagtcaaccaatcatgaaggatcttg,mergedseq5,barcode01,/mnt/c/Users/mftor/Documents/metabarcoding/dat...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
